# 🚀 Architecture Explorer: Matrix Multiplication
*Compare how Compilers and Hardware Architecture impact DGEMM performance.*

**Instructions:**
1. Run the **Source Code** cell to save the kernel.
2. Run the **Benchmark** cell to see the performance results.

In [ ]:
#@title 🚀 Run Benchmark { display-mode: "form" }

#@markdown ### 1. Hardware & Compiler Settings
matrix_size = 1024 #@param [512, 1024, 2048] {type:"raw"}
optimization_level = "-O3" #@param ["-O0", "-O1", "-O2", "-O3"]
# This is the "friendly" upgrade:
optimize_for_this_cpu = True #@param {type:"boolean"} #@title Optimize for current hardware? (Recommended)

#@markdown ### 2. Algorithmic Impact
#@markdown *Row-major is typically "Cache-Friendly" for C/C++ loops.*
memory_layout = "Row-major (Fast)" #@param ["Row-major (Fast)", "Column-major (Slow)"]

import subprocess

# Define the indexing logic based on selection
is_row_major = "Row-major" in memory_layout
layout_str = "Row-major" if is_row_major else "Column-major"

# Update C code to swap indices based on layout
c_code = f"""
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

void dgemm (size_t n, double* A, double* B, double* C) {{
    for (size_t i = 0; i < n; ++i)
        for (size_t k = 0; k < n; ++k) {{
            double aik = A[i*n+k]; 
            for(size_t j = 0; j < n; ++j) {{
                // This is where the magic happens:
                {"C[i*n+j] += aik * B[k*n+j];" if is_row_major else "C[j*n+i] += aik * B[j*n+k];"}
            }}
        }}
}}

int main() {{
    const int n = {matrix_size};
    double *a = (double *)malloc(n * n * sizeof(double));
    double *b = (double *)malloc(n * n * sizeof(double));
    double *c = (double *)malloc(n * n * sizeof(double));

    for(int i=0; i<n*n; i++) {{ a[i]=1.0; b[i]=2.0; c[i]=0.0; }}

    struct timespec start, end;
    clock_gettime(CLOCK_MONOTONIC, &start);
    dgemm(n, a, b, c);
    clock_gettime(CLOCK_MONOTONIC, &end);

    double time = (end.tv_sec - start.tv_sec) + (end.tv_nsec - start.tv_nsec) / 1e9;
    printf("Layout: %s | Matrix: %d | Time: %.4f s\\n", "{layout_str}", n, time);

    free(a); free(b); free(c);
    return 0;
}}
"""

with open("dgemm.c", "w") as f:
    f.write(c_code)

cmd = ["gcc", optimization_level, "dgemm.c", "-o", "benchmark"]
if use_native_march: cmd.append("-march=native")

compile_res = subprocess.run(cmd, capture_output=True, text=True)
if compile_res.returncode == 0:
    run_res = subprocess.run(["./benchmark"], capture_output=True, text=True)
    print(run_res.stdout)
else:
    print(f"❌ Error: {compile_res.stderr}")

🔨 Compiling with: gcc -O3 dgemm.c -o benchmark -march=native...
✅ Compilation Successful. Running...
Matrix Size: 1024 | Time: 7.4868 seconds

